In [ ]:
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("historical_data.csv",nrows = 50000)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.drop_duplicates()

In [ ]:
df['Execution Price'] = df['Execution Price'].astype(str).str.replace('@','').astype(float)

In [ ]:
df['Timestamp IST'] = pd.to_datetime(df['Timestamp IST'], dayfirst=True)

In [ ]:
df.dropna(inplace=True)

In [ ]:
df['hour'] = df['Timestamp IST'].dt.hour
df['date'] = df['Timestamp IST'].dt.date

In [ ]:
df['is_profit'] = df['Closed PnL'] > 0

In [ ]:
df['Closed PnL'].sum()

In [ ]:
 df['Fee'].sum()

In [ ]:
coin_profit = df.groupby('Coin')['Closed PnL'].sum().sort_values(ascending=False)
print(coin_profit.head(10))

In [ ]:
coin_profit.tail(10)

In [ ]:
df['Side'].value_counts()

In [ ]:
df.groupby('hour')['Closed PnL'].sum()

In [ ]:
df = df.sort_values('Timestamp IST')
df['cum_pnl'] = df['Closed PnL'].cumsum()

plt.plot(df['Timestamp IST'], df['cum_pnl'])
plt.title("Cumulative Profit Over Time")
plt.show()

In [ ]:
coin_profit.head(10).plot(kind='bar')
plt.title("Top Profitable Coins")
plt.show()

In [ ]:
sns.histplot(df['Closed PnL'], bins=50)
plt.title("Distribution of Profit & Loss")
plt.show()

In [ ]:
df['Side'].value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.title("Buy vs Sell Distribution")
plt.ylabel('')
plt.show()

In [ ]:
worst_coins = df.groupby('Coin')['Closed PnL'].sum().sort_values().head(10)

worst_coins.plot(kind='bar')
plt.title("Top 10 Loss Making Coins")
plt.show()

In [ ]:
hourly_pnl = df.groupby('hour')['Closed PnL'].sum()

hourly_pnl.plot(kind='line', marker='o')
plt.title("Profit by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("PnL")
plt.show()

In [ ]:
plt.scatter(df['Fee'], df['Closed PnL'])
plt.title("Fee vs Profit")
plt.xlabel("Fee")
plt.ylabel("PnL")
plt.show()

In [ ]:
pivot = df.pivot_table(values='Closed PnL', index='hour', columns='Side', aggfunc='sum')

sns.heatmap(pivot, annot=True, cmap='coolwarm')
plt.title("Profit Heatmap (Hour vs Buy/Sell)")
plt.show()

In [ ]:
features = [
    'Execution Price',
    'Size Tokens',
    'Size USD',
    'Start Position',
    'Fee',
    'hour',
    'Coin',
    'Side',
    'Direction',
    'Crossed'
]

In [ ]:
X = df[features]
y = df['Closed PnL']

In [ ]:
num_col = [
    'Execution Price',
    'Size Tokens',
    'Size USD',
    'Start Position',
    'Fee',
    'hour'
]

cat_col = [
    'Coin',
    'Side',
    'Direction'
]

bool_col = ['Crossed']

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

num_trans = StandardScaler()

cat_trans = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_trans, num_col),
        ('cat', cat_trans, cat_col),
        ('bool', 'passthrough', bool_col)
    ]
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import r2_score

y_pred = pipeline.predict(X_test)

print("R2 Score:", r2_score(y_test, y_pred))